# 3. Bias and gate sweeps

**Learning goals.** In this tutorial you will:

- understand what a charge-stability diagram represents;
- update a QmeQ system without rebuilding it at every point;
- calculate current and differential conductance maps; and
- recognize numerical checks needed for parameter sweeps.

We use the spinful Anderson model introduced in Tutorial 2.

## What is a stability diagram?

A gate voltage shifts the orbital energy $\varepsilon$, while a bias voltage separates the lead chemical potentials, $\mu_L=V/2$ and $\mu_R=-V/2$. A map of current or differential conductance versus $(V,\varepsilon)$ reveals the energies required to add and remove electrons.

In a sequential-tunnelling picture, the boundaries occur when a dot transition energy crosses a lead chemical potential. Around the singly occupied region, these boundaries enclose a low-current Coulomb diamond.

**Prediction before calculating.** The current must reverse sign under $V\rightarrow -V$ for this left-right-symmetric model. Around $\varepsilon=-U/2$ and small bias, the current should be suppressed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qmeq

U = 4.0
temperature = 0.2
gamma = 0.1
bandwidth = 40.0
tunnel_amplitude = np.sqrt(gamma / (2 * np.pi))

system = qmeq.Builder(
    nsingle=2,
    hsingle={(0, 0): -U / 2, (1, 1): -U / 2},
    coulomb={(0, 1, 1, 0): U},
    nleads=4,
    tleads={(0, 0): tunnel_amplitude, (1, 0): tunnel_amplitude,
            (2, 1): tunnel_amplitude, (3, 1): tunnel_amplitude},
    mulst={0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0},
    tlst={0: temperature, 1: temperature, 2: temperature, 3: temperature},
    dband=bandwidth,
    kerntype="Pauli",
)

## Reusing a system safely

`system.change(...)` updates model parameters. After changing the dot Hamiltonian, `solve(masterq=False)` diagonalizes the dot without solving the transport problem. For each subsequent bias value, `solve(qdq=False)` reuses that diagonalization and solves only the master equation.

These flags are useful in sweeps, but they carry responsibility: `qdq=False` is correct only when the dot Hamiltonian has not changed since the last diagonalization.

In [ ]:
# Modest grids keep this tutorial quick. Increase them for publication plots.
gate_values = np.linspace(-1.5 * U, 0.5 * U, 41)
bias_values = np.linspace(-2.0 * U, 2.0 * U, 51)
current = np.empty((len(bias_values), len(gate_values)))
conservation_error = np.empty_like(current)

for gate_index, gate in enumerate(gate_values):
    system.change(hsingle={(0, 0): gate, (1, 1): gate})
    system.solve(masterq=False)

    for bias_index, bias in enumerate(bias_values):
        system.change(
            mulst={0: bias / 2, 1: -bias / 2, 2: bias / 2, 3: -bias / 2}
        )
        system.solve(qdq=False)
        current[bias_index, gate_index] = system.current[0] + system.current[2]
        conservation_error[bias_index, gate_index] = abs(np.sum(system.current))

assert np.max(conservation_error) < 1e-10
print(f"largest current-conservation error: {np.max(conservation_error):.2e}")

Differential conductance is $G=\partial I/\partial V$. `numpy.gradient` estimates it from the already calculated current grid. This avoids a second master-equation solve at a tiny displaced bias, but the result depends on the bias spacing.

In [ ]:
conductance = np.gradient(current, bias_values, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
extent = [gate_values[0] / U, gate_values[-1] / U,
          bias_values[0] / U, bias_values[-1] / U]

current_plot = axes[0].imshow(
    current / gamma, origin="lower", aspect="auto", extent=extent, cmap="coolwarm"
)
fig.colorbar(current_plot, ax=axes[0], label="$I_L/\\Gamma$")
axes[0].set_title("Particle current")

conductance_plot = axes[1].imshow(
    conductance, origin="lower", aspect="auto", extent=extent, cmap="viridis"
)
fig.colorbar(conductance_plot, ax=axes[1], label="$\\partial I_L/\\partial V$")
axes[1].set_title("Differential conductance")

for axis in axes:
    axis.set_xlabel("$\\varepsilon/U$")
axes[0].set_ylabel("$V/U$")
fig.tight_layout()

The low-current diamond centered at $\varepsilon=-U/2$ is the Coulomb-blockaded region. Its edges mark alignment of an addition or removal energy with a lead chemical potential. Conductance emphasizes these thresholds.

## Symmetry and grid checks

A two-dimensional plot can look plausible even when a calculation is wrong. Test model symmetries and numerical resolution directly.

In [ ]:
# For symmetric couplings, current is odd in bias.
oddness_error = np.max(np.abs(current + current[::-1, :]))
assert oddness_error < 1e-10

# Equilibrium is the central row because the bias grid contains zero.
zero_bias_index = np.argmin(np.abs(bias_values))
assert np.allclose(current[zero_bias_index], 0.0, atol=1e-12)

print(f"largest bias-antisymmetry error: {oddness_error:.2e}")

## Interpreting numerical structure

- A coarse gate or bias grid makes threshold lines jagged.
- Numerical differentiation amplifies grid noise; compare at least two spacings before trusting fine conductance features.
- A larger plot grid improves resolution, not physical accuracy. Accuracy also depends on whether the selected master equation is valid.
- The Pauli result predicts zero sequential current deep inside a Coulomb diamond up to thermal activation. Cotunnelling can change that conclusion.

## Exercises

1. Repeat the calculation with 21 and 81 bias points. Which features of the current are stable, and which features of the numerical derivative move?
2. Break left-right symmetry by reducing the two right tunnelling amplitudes. The current need no longer be exactly odd in bias; explain why the symmetry check must match the model.
3. Increase the temperature. Predict how the diamond edges and zero-bias conductance change.